# 2-D Tubular Flow (Cylindrical Coordinates - Steady State)

### Problem Description

This notebook simulates **steady-state flow** in a cylindrical tube, a fundamental problem in fluid mechanics with applications in:
- Pipeline flow systems
- Blood flow in vessels
- Microfluidic devices
- Heat exchanger tubes

### Physical System

A cylindrical tube where:
- **Flow**: Axisymmetric laminar flow (Hagen-Poiseuille)
- **Geometry**: Cylindrical tube with radius R and length L
- **Coordinates**: Cylindrical (r, z) with axial symmetry (no θ dependence)

### Governing Equations (Cylindrical Coordinates - Steady State)

#### 1. Continuity Equation (Incompressible, Axisymmetric)
$$\frac{1}{r}\frac{\partial(rv_r)}{\partial r} + \frac{\partial v_z}{\partial z} = 0$$

#### 2. Navier-Stokes Equations (Steady State, Axisymmetric)

**r-momentum:**
$$\rho\left(v_r\frac{\partial v_r}{\partial r} + v_z\frac{\partial v_r}{\partial z}\right) = -\frac{\partial p}{\partial r} + \mu\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial v_r}{\partial r}\right) - \frac{v_r}{r^2} + \frac{\partial^2 v_r}{\partial z^2}\right]$$

**z-momentum:**
$$\rho\left(v_r\frac{\partial v_z}{\partial r} + v_z\frac{\partial v_z}{\partial z}\right) = -\frac{\partial p}{\partial z} + \mu\left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial v_z}{\partial r}\right) + \frac{\partial^2 v_z}{\partial z^2}\right]$$

where:
- $v_r, v_z$ = velocity components in r and z directions (m/s)
- $p$ = pressure (Pa)
- $\rho$ = fluid density (kg/m³)
- $\mu$ = dynamic viscosity (Pa·s)

### Analytical Solution for Fully Developed Flow

For fully developed laminar flow in a circular tube (Hagen-Poiseuille):

$$v_z(r) = v_{z,max}\left(1 - \frac{r^2}{R^2}\right) = 2v_{avg}\left(1 - \frac{r^2}{R^2}\right)$$

where $v_{z,max} = 2v_{avg}$ and $v_r = 0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 1. Physical Parameters and Tube Geometry

In [ ]:
# Tube geometry
R = 0.05              # Tube radius (5 cm)
L = 2.0               # Tube length (2 m)

# Fluid properties (typical gas at moderate conditions)
rho = 1.2             # Density (kg/m³) - gas
mu = 1.8e-5           # Dynamic viscosity (Pa·s) - air-like
nu = mu / rho         # Kinematic viscosity (m²/s)

# Flow parameters
v_avg = 0.05           # Average axial velocity (m/s)
v_max = 2 * v_avg     # Maximum velocity (centerline)
Q = np.pi * R**2 * v_avg  # Volumetric flow rate (m³/s)
Re = rho * v_avg * (2*R) / mu  # Reynolds number (based on diameter)

print("=" * 70)
print("TUBULAR FLOW SIMULATION - CYLINDRICAL COORDINATES (STEADY STATE)")
print("=" * 70)
print(f"\nTube Geometry:")
print(f"  Tube radius (R): {R*100:.1f} cm")
print(f"  Tube length (L): {L:.2f} m")
print(f"  L/D ratio: {L/(2*R):.1f}")
print(f"\nFluid Properties:")
print(f"  Density: {rho:.2f} kg/m³")
print(f"  Viscosity: {mu*1e6:.2f} μPa·s")
print(f"  Kinematic viscosity: {nu:.2e} m²/s")
print(f"\nFlow Conditions:")
print(f"  Average velocity: {v_avg:.2f} m/s")
print(f"  Maximum velocity: {v_max:.2f} m/s")
print(f"  Volumetric flow rate: {Q*1e6:.2f} L/min")
print(f"  Reynolds number (Re): {Re:.1f}")
print(f"\nFlow Regime: {'Laminar' if Re < 2300 else 'Turbulent'}")
print("=" * 70)

## 2. Numerical Grid Setup

In [ ]:
# Grid parameters
nr = 50               # Number of grid points in radial direction
nz = 100              # Number of grid points in axial direction

# Create grid
r = np.linspace(0, R, nr)        # Radial coordinate (0 to R)
z = np.linspace(0, L, nz)        # Axial coordinate (0 to L)
dr = r[1] - r[0]
dz = z[1] - z[0]

# Create meshgrid
Z, R_grid = np.meshgrid(z, r)

print(f"\nGrid Setup:")
print(f"  Radial points (nr): {nr}")
print(f"  Axial points (nz): {nz}")
print(f"  Grid spacing: Δr = {dr*1000:.2f} mm, Δz = {dz*100:.2f} cm")
print(f"  Total grid points: {nr * nz}")

## 3. Velocity Field - Hagen-Poiseuille Flow

For fully developed laminar flow in a circular tube:
- Axial velocity has parabolic profile
- Radial velocity is zero
- Flow is independent of z (fully developed)

In [ ]:
# Initialize velocity fields
v_r = np.zeros((nr, nz))         # Radial velocity (zero for fully developed flow)
v_z = np.zeros((nr, nz))         # Axial velocity

# Compute axial velocity - Hagen-Poiseuille profile
for i in range(nr):
    v_z[i, :] = v_max * (1 - (r[i]/R)**2)

# Verify flow rate
# Q = ∫₀ᴿ v_z(r) · 2πr dr
Q_numerical = 2 * np.pi * np.trapz(r * v_z[:, 0], r)
Q_error = abs(Q_numerical - Q) / Q * 100

print(f"\nVelocity Field Verification:")
print(f"  Analytical flow rate: {Q*1e6:.4f} L/min")
print(f"  Numerical flow rate: {Q_numerical*1e6:.4f} L/min")
print(f"  Error: {Q_error:.3f}%")
print(f"  Maximum velocity: {np.max(v_z):.4f} m/s")
print(f"  Minimum velocity: {np.min(v_z):.4f} m/s")

## 4. Visualization of Velocity Field

In [ ]:
# Create comprehensive velocity visualization
fig = plt.figure(figsize=(16, 12))

# 1. 2D Contour plot of axial velocity
ax1 = plt.subplot(3, 2, 1)
contour = ax1.contourf(Z, R_grid*100, v_z, levels=20, cmap='viridis')
ax1.set_xlabel('Axial Position, z (m)', fontsize=11)
ax1.set_ylabel('Radial Position, r (cm)', fontsize=11)
ax1.set_title('Axial Velocity Field, $v_z(r,z)$', fontsize=12, fontweight='bold')
cbar1 = plt.colorbar(contour, ax=ax1)
cbar1.set_label('Velocity (m/s)', fontsize=10)

# 2. Radial velocity profile at different axial positions
ax2 = plt.subplot(3, 2, 2)
ax2.plot(r*100, v_z[:, 0], 'b-', linewidth=2.5, label='Numerical')
# Overlay analytical solution
r_analytical = np.linspace(0, R, 200)
v_analytical = v_max * (1 - (r_analytical/R)**2)
ax2.plot(r_analytical*100, v_analytical, 'r--', linewidth=2, label='Analytical')
ax2.set_xlabel('Radial Position, r (cm)', fontsize=11)
ax2.set_ylabel('Axial Velocity (m/s)', fontsize=11)
ax2.set_title('Velocity Profile (Hagen-Poiseuille)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. 3D surface plot
ax3 = plt.subplot(3, 2, 3, projection='3d')
surf = ax3.plot_surface(Z, R_grid*100, v_z, cmap='plasma',
                        edgecolor='none', alpha=0.9, antialiased=True)
ax3.set_xlabel('z (m)', fontsize=10)
ax3.set_ylabel('r (cm)', fontsize=10)
ax3.set_zlabel('$v_z$ (m/s)', fontsize=10)
ax3.set_title('3D Velocity Surface', fontsize=12, fontweight='bold')
ax3.view_init(elev=25, azim=45)

# 4. Streamlines
ax4 = plt.subplot(3, 2, 4)
# Create streamlines (since v_r = 0, they are just horizontal lines)
Y_stream = np.linspace(0, R, 10)
for y in Y_stream:
    ax4.plot([0, L], [y*100, y*100], 'b-', alpha=0.6, linewidth=1.5)
ax4.set_xlabel('Axial Position, z (m)', fontsize=11)
ax4.set_ylabel('Radial Position, r (cm)', fontsize=11)
ax4.set_title('Flow Streamlines', fontsize=12, fontweight='bold')
ax4.set_ylim([0, R*100])
ax4.grid(True, alpha=0.3)

# 5. Velocity magnitude vs radius (normalized)
ax5 = plt.subplot(3, 2, 5)
r_normalized = r / R
v_normalized = v_z[:, 0] / v_max
ax5.plot(r_normalized, v_normalized, 'b-', linewidth=2.5)
ax5.fill_between(r_normalized, 0, v_normalized, alpha=0.3)
ax5.set_xlabel('Normalized Radius, r/R', fontsize=11)
ax5.set_ylabel('Normalized Velocity, $v_z/v_{max}$', fontsize=11)
ax5.set_title('Normalized Velocity Profile', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.set_xlim([0, 1])
ax5.set_ylim([0, 1.1])

# 6. Velocity gradient (shear rate)
ax6 = plt.subplot(3, 2, 6)
dvz_dr = np.gradient(v_z[:, nz//2], dr)
shear_rate = np.abs(dvz_dr)
ax6.plot(r*100, shear_rate, 'r-', linewidth=2.5)
ax6.set_xlabel('Radial Position, r (cm)', fontsize=11)
ax6.set_ylabel('Shear Rate, |dv$_z$/dr| (1/s)', fontsize=11)
ax6.set_title('Velocity Gradient (Shear Rate)', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Flow Characteristics Analysis

In [ ]:
# Calculate key flow characteristics

# 1. Wall shear stress
# τ_w = μ (dv_z/dr)|_{r=R}
dvz_dr_wall = (v_z[-2, 0] - v_z[-1, 0]) / dr
tau_wall = mu * abs(dvz_dr_wall)

# Analytical wall shear stress
tau_wall_analytical = 4 * mu * v_avg / R

# 2. Pressure drop (Hagen-Poiseuille equation)
# Δp/L = 8μv_avg/R²
dp_dz = 8 * mu * v_avg / R**2
delta_p = dp_dz * L

# 3. Friction factor
# f = 64/Re (for laminar flow)
f = 64 / Re

# 4. Entrance length (laminar flow)
# L_e ≈ 0.05 × Re × D
L_entrance = 0.05 * Re * (2*R)

# 5. Maximum shear stress at wall
max_shear_stress = mu * 2 * v_max / R

print("\n" + "="*70)
print("FLOW CHARACTERISTICS ANALYSIS")
print("="*70)
print(f"\nWall Shear Stress:")
print(f"  Numerical: {tau_wall:.6f} Pa")
print(f"  Analytical: {tau_wall_analytical:.6f} Pa")
print(f"  Error: {abs(tau_wall - tau_wall_analytical)/tau_wall_analytical * 100:.2f}%")
print(f"\nPressure Drop:")
print(f"  Pressure gradient: {dp_dz:.4f} Pa/m")
print(f"  Total pressure drop: {delta_p:.4f} Pa")
print(f"\nFriction Factor:")
print(f"  Darcy friction factor: {f:.4f}")
print(f"  Fanning friction factor: {f/4:.4f}")
print(f"\nEntry Length:")
print(f"  Hydrodynamic entrance length: {L_entrance:.3f} m")
print(f"  L/L_e ratio: {L/L_entrance:.2f}")
print(f"\nShear Characteristics:")
print(f"  Maximum shear rate: {max_shear_stress/mu:.2f} s⁻¹")
print(f"  Wall shear rate: {2*v_avg/R:.2f} s⁻¹")
print("="*70)

## 6. Velocity Distribution Statistics

In [ ]:
# Statistical analysis of velocity field
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# 1. Velocity histogram
ax1.hist(v_z.flatten(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(v_avg, color='red', linestyle='--', linewidth=2, label=f'Average: {v_avg:.2f} m/s')
ax1.axvline(v_max, color='green', linestyle='--', linewidth=2, label=f'Maximum: {v_max:.2f} m/s')
ax1.set_xlabel('Velocity (m/s)', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Velocity Distribution', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. Volumetric flow rate distribution
# Calculate flow rate in annular elements
Q_elements = np.zeros(nr-1)
for i in range(nr-1):
    r_mid = (r[i] + r[i+1]) / 2
    v_mid = (v_z[i, 0] + v_z[i+1, 0]) / 2
    Q_elements[i] = v_mid * 2 * np.pi * r_mid * dr

r_mid_plot = (r[:-1] + r[1:]) / 2
ax2.plot(r_mid_plot*100, Q_elements*1e6, 'b-', linewidth=2.5)
ax2.fill_between(r_mid_plot*100, 0, Q_elements*1e6, alpha=0.3)
ax2.set_xlabel('Radial Position, r (cm)', fontsize=11)
ax2.set_ylabel('Annular Flow Rate (L/min)', fontsize=11)
ax2.set_title('Volumetric Flow Rate Distribution', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Kinetic energy distribution
KE_per_volume = 0.5 * rho * v_z[:, 0]**2  # J/m³
ax3.plot(r*100, KE_per_volume, 'r-', linewidth=2.5)
ax3.fill_between(r*100, 0, KE_per_volume, alpha=0.3, color='red')
ax3.set_xlabel('Radial Position, r (cm)', fontsize=11)
ax3.set_ylabel('Kinetic Energy Density (J/m³)', fontsize=11)
ax3.set_title('Kinetic Energy Distribution', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Momentum flux distribution
momentum_flux = rho * v_z[:, 0]**2  # kg/(m·s²)
ax4.plot(r*100, momentum_flux, 'g-', linewidth=2.5)
ax4.fill_between(r*100, 0, momentum_flux, alpha=0.3, color='green')
ax4.set_xlabel('Radial Position, r (cm)', fontsize=11)
ax4.set_ylabel('Momentum Flux (kg/(m·s²))', fontsize=11)
ax4.set_title('Momentum Flux Distribution', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nVelocity Field Statistics:")
print(f"  Mean velocity: {np.mean(v_z):.4f} m/s")
print(f"  Median velocity: {np.median(v_z):.4f} m/s")
print(f"  Standard deviation: {np.std(v_z):.4f} m/s")
print(f"  Velocity range: {np.min(v_z):.4f} to {np.max(v_z):.4f} m/s")

## 7. Summary and Applications

### Key Findings:

1. **Flow Pattern**:
   - Parabolic velocity profile (Hagen-Poiseuille)
   - Maximum velocity at centerline = 2 × average velocity
   - Laminar flow regime (Re << 2300)
   - Zero radial velocity (axisymmetric, fully developed)

2. **Velocity Distribution**:
   - Highest velocity at tube center
   - Decreases quadratically toward wall
   - No-slip condition at wall (v = 0 at r = R)
   - Fully developed flow throughout tube length

3. **Flow Characteristics**:
   - Wall shear stress proportional to average velocity
   - Pressure drop linear with tube length
   - Friction factor inversely proportional to Re
   - Entrance length typically small for low Re

### Engineering Applications:

#### 1. **Pipeline Transport**
   - Oil and gas pipelines
   - Water distribution systems
   - Chemical process piping

#### 2. **Biomedical Engineering**
   - Blood flow in arteries and veins
   - Drug delivery systems
   - Microfluidic devices

#### 3. **Heat Exchangers**
   - Tube-side flow in shell-and-tube exchangers
   - Condensers and evaporators
   - Heat transfer calculations

#### 4. **Manufacturing Processes**
   - Polymer extrusion
   - Coating applications
   - Ink-jet printing

### Design Considerations:

1. **Pressure Drop Minimization**:
   - Increase tube diameter to reduce Δp
   - Keep Reynolds number in laminar regime
   - Minimize tube length when possible

2. **Flow Uniformity**:
   - Consider entrance effects
   - Account for developing flow region
   - Use flow straighteners if needed

3. **Shear Stress Management**:
   - Critical for biological applications
   - Important for polymer processing
   - Affects wall erosion and fouling

## 8. Exercises

**Exercise 1**: Investigate the effect of Reynolds number on the velocity profile. Increase the flow velocity and observe how the profile develops.

**Exercise 2**: Compare the pressure drop for laminar flow (Re = 1000) versus turbulent flow (Re = 10000). Use appropriate friction factor correlations.

**Exercise 3**: Calculate the total kinetic energy in the flow and compare it to the pumping power required to overcome pressure drop.

**Exercise 4**: For a blood vessel application (ρ = 1060 kg/m³, μ = 3.5×10⁻³ Pa·s), determine the appropriate flow conditions to maintain Re < 1000.

In [ ]:
# Your code here for exercises


## References

1. Bird, R. B., Stewart, W. E., & Lightfoot, E. N. (2007). *Transport Phenomena* (2nd ed.). John Wiley & Sons.

2. White, F. M. (2016). *Fluid Mechanics* (8th ed.). McGraw-Hill Education.

3. Munson, B. R., Young, D. F., & Okiishi, T. H. (2013). *Fundamentals of Fluid Mechanics* (7th ed.). John Wiley & Sons.

4. Fox, R. W., McDonald, A. T., & Pritchard, P. J. (2011). *Introduction to Fluid Mechanics* (8th ed.). John Wiley & Sons.

5. Schlichting, H., & Gersten, K. (2017). *Boundary Layer Theory* (9th ed.). Springer.